In [8]:
# ============================================================
# NOTEBOOK 7 — STRATEGIC TARGET SEGMENT
# Combine priority_level with demographic and behaviour signals
# to produce 11 mutually exclusive, exhaustive strategic segments.
#
# ARCHITECTURE — read this carefully:
# 1. priority_level was assigned in Notebook 4 from behaviour_score.
# 2. Rules below operate STRICTLY WITHIN priority tier — no rule
#    can move a customer from one tier into another. The function
#    receives the customer's priority_level first, then within that
#    tier applies sub-rules in defined order, with a positively-
#    defined fallback bucket per tier.
# 3. After assignment we ASSERT no tier leakage — every "Protect"
#    customer is High Priority, every "Grow" / "Cross-Sell" is
#    Growth Priority, every "Maintain" / "Premium" / "Upsell" is
#    Maintain, every "Reactivate" is Low Priority.
#
# This architecture is the explicit fix for the failure mode in
# the previous segmentation, where behaviour rules silently 
# overrode priority assignments.
# ============================================================

In [9]:
import pandas as pd
import numpy as np
import os

In [10]:
# ------------------------------------------------------------
# PATH SETUP & LOAD CHECKPOINT
# ------------------------------------------------------------

In [11]:
path = r'/Users/elia/Desktop/[02] Data Projects/[2] Working Folder/[3] InstaCart Store'

df_customers_agg = pd.read_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'customers_agg_step6.pkl')
)
print(f"Checkpoint loaded: {df_customers_agg.shape}")
print()

Checkpoint loaded: (206209, 26)



In [12]:
# ============================================================
# 7A — STRATEGIC TARGET SEGMENT FUNCTION
# ============================================================
# Function dispatches FIRST on priority_level, then within tier.
# Every branch returns a label — no None, no fall-through.

In [13]:
def assign_strategic_segment(row):
    """
    Assign a strategic target segment using strict tier-first logic.
    
    Architecture:
        1. Read priority_level (already assigned from behaviour_score).
        2. Within tier, evaluate sub-rules in declared order.
        3. Each tier has a positively-defined residual segment that
           captures customers not matching the more specific rules.
    
    Returns one of 11 segment labels. Function is exhaustive and
    no rule can move a customer across priority tiers.
    """
    priority = row['priority_level']
    income   = row['income_group']
    house    = row['household_group']
    price_r  = row['price_pct_rank']
    orders_r = row['orders_pct_rank']
    
    # ============================================================
    # TIER 1 — HIGH PRIORTY (top 12% by behaviour_score)
    # Strategy: PROTECT. Two segments split by income.
    # ============================================================
    if priority == 'High Priority':
        # Affluent retention — premium loyalty programmes, white-glove service
        if income in ('High Income', 'Very High Income'):
            return 'Protect High-Value Repeat Customers'
        # Engagement-protect — for elite behaviour without elite wallet
        # POSITIVE DEFINITION: High Priority customers with Low or
        # Middle Income whose retention value is engagement, not spend.
        return 'Protect Highly Engaged Customers'
    
    # ============================================================
    # TIER 2 — GROWTH PRIORITY (next 24%)
    # Strategy: GROW. Four segments split by income and household.
    # Priority order:
    #   1. Very High Income → Affluent Active (premium upsell)
    #   2. Family Household + non-Very-High → Active Family (bulk/family marketing)
    #   3. Low Income + non-Family → Value-Conscious (coupon/value marketing)
    #   4. Everything else → Cross-Sell Broad Basket Explorers (catalogue breadth)
    # ============================================================
    if priority == 'Growth Priority':
        if income == 'Very High Income':
            return 'Grow Affluent Active Customers'
        if house == 'Family Household' and income in ('Low Income', 'Middle Income', 'High Income'):
            return 'Grow Active Family Shoppers'
        if income == 'Low Income' and house != 'Family Household':
            return 'Grow Active Value-Conscious Customers'
        # POSITIVE DEFINITION: Growth-tier customers who are non-Family,
        # non-Very-High income, and non-Low-income — typically singles
        # or small households (Single No Kids, Divorced/Widowed, Living 
        # with Family) on Middle or High Income. Growth lever for this
        # group is catalogue breadth: introduce them to new categories
        # and SKUs they haven't yet explored.
        return 'Cross-Sell Broad Basket Explorers'
    
    # ============================================================
    # TIER 3 — MAINTAIN (middle 26%)
    # Strategy: HOLD. Three sub-segments by price and orders signal.
    # Thresholds (0.70 / 0.50 / 0.40) validated against in-tier
    # distribution: produce 30% / 24% / 45% split, all campaign-viable.
    # ============================================================
    if priority == 'Maintain':
        # Premium-leaning: above 70th percentile of price rank.
        # That's where price preference becomes a consistent signal
        # in this dataset — below it, preference is too noisy to act on.
        if price_r >= 0.70:
            return 'Premium Higher Price Buyers'
        # Frequent shoppers with low price points — natural cross-
        # promotion target for premium-tier upsell.
        if orders_r >= 0.50 and price_r < 0.40:
            return 'Upsell Frequent Low-Price Buyers'
        # POSITIVE DEFINITION: Maintain-tier customers without strong
        # price or frequency signal — the genuine middle. Strategy
        # is hold-position, not intervene.
        return 'Maintain Stable Mid-Value Customers'
    
    # ============================================================
    # TIER 4 — LOW PRIORITY (bottom 38%)
    # Strategy: REACTIVATE or accept loss.
    # Families get a distinct campaign because higher potential basket.
    # ============================================================
    if priority == 'Low Priority':
        if house == 'Family Household':
            return 'Reactivate Low-Engagement Families'
        # POSITIVE DEFINITION: Low-Priority customers who are Single No
        # Kids, Divorced/Widowed, or Living with Family. General-purpose
        # reactivation pool — typically smaller potential basket size.
        return 'Reactivate Low-Engagement Customers'
    
    # Should be unreachable — every priority_level value handled above.
    # If we reach this line, priority_level has an unexpected value
    # and the assertion on null segments will catch it.
    return None

In [14]:
# ============================================================
# 7B — APPLY FUNCTION
# ============================================================

df_customers_agg['strategic_target_segment'] = df_customers_agg.apply(
    assign_strategic_segment, axis=1
)

# Set as ordered categorical with strategic grouping order
strategic_segment_order = [
    # High Priority (Protect)
    'Protect High-Value Repeat Customers',
    'Protect Highly Engaged Customers',
    # Growth Priority (Grow / Cross-Sell)
    'Grow Affluent Active Customers',
    'Grow Active Family Shoppers',
    'Grow Active Value-Conscious Customers',
    'Cross-Sell Broad Basket Explorers',
    # Maintain
    'Premium Higher Price Buyers',
    'Upsell Frequent Low-Price Buyers',
    'Maintain Stable Mid-Value Customers',
    # Low Priority (Reactivate)
    'Reactivate Low-Engagement Families',
    'Reactivate Low-Engagement Customers'
]

df_customers_agg['strategic_target_segment'] = pd.Categorical(
    df_customers_agg['strategic_target_segment'],
    categories=strategic_segment_order,
    ordered=True
)

In [15]:
# ============================================================
# 7C — VALIDATION (the critical part)
# ============================================================

# Validation 1: no nulls — function must be exhaustive
assert df_customers_agg['strategic_target_segment'].isnull().sum() == 0, \
    "Some customers have no strategic_target_segment — function is not exhaustive"

# Validation 2: every assigned label must be in the expected list
expected = set(strategic_segment_order)
actual   = set(df_customers_agg['strategic_target_segment'].astype(str).unique())
assert actual.issubset(expected), \
    f"Unexpected segment values: {actual - expected}"

# Validation 3 — THE CRITICAL ONE: STRICT TIER LEAKAGE CHECK
# This is the explicit guard against the failure mode in the previous
# segmentation. Every segment name maps to exactly one priority tier;
# we assert that mapping holds in the data.
tier_segment_mapping = {
    'High Priority':   ['Protect High-Value Repeat Customers',
                        'Protect Highly Engaged Customers'],
    'Growth Priority': ['Grow Affluent Active Customers',
                        'Grow Active Family Shoppers',
                        'Grow Active Value-Conscious Customers',
                        'Cross-Sell Broad Basket Explorers'],
    'Maintain':        ['Premium Higher Price Buyers',
                        'Upsell Frequent Low-Price Buyers',
                        'Maintain Stable Mid-Value Customers'],
    'Low Priority':    ['Reactivate Low-Engagement Families',
                        'Reactivate Low-Engagement Customers']
}

for tier, allowed_segments in tier_segment_mapping.items():
    customers_in_tier = df_customers_agg[df_customers_agg['priority_level'] == tier]
    segments_in_tier  = set(customers_in_tier['strategic_target_segment'].astype(str).unique())
    illegal_segments  = segments_in_tier - set(allowed_segments)
    assert len(illegal_segments) == 0, \
        f"⚠️ TIER LEAKAGE: {tier} customers have illegal segments: {illegal_segments}"

# Reverse check: each segment's customers must all be in its declared tier
segment_to_tier = {seg: tier
                   for tier, segs in tier_segment_mapping.items()
                   for seg in segs}

for segment, expected_tier in segment_to_tier.items():
    customers_in_seg = df_customers_agg[
        df_customers_agg['strategic_target_segment'] == segment
    ]
    tiers_in_seg = set(customers_in_seg['priority_level'].astype(str).unique())
    if len(customers_in_seg) > 0:
        assert tiers_in_seg == {expected_tier}, \
            f"⚠️ TIER LEAKAGE: {segment} has customers in tiers {tiers_in_seg}, expected only {expected_tier}"

# Validation 4: total count preserved
assert df_customers_agg.shape[0] == 206209

print("✅ All Notebook 7 assertions passed")
print("✅ STRICT TIERING CONFIRMED — no behaviour rule leaked across priority tiers")
print()

✅ All Notebook 7 assertions passed
✅ STRICT TIERING CONFIRMED — no behaviour rule leaked across priority tiers



In [16]:
# ============================================================
# 7D — DESCRIPTIVE OUTPUT
# ============================================================

print("Strategic target segment distribution:")
seg_counts = df_customers_agg['strategic_target_segment'].value_counts().reindex(strategic_segment_order)
seg_pct    = (df_customers_agg['strategic_target_segment'].value_counts(normalize=True) * 100).reindex(strategic_segment_order)
for seg in strategic_segment_order:
    print(f"  {seg:<45} {seg_counts[seg]:>7,}  ({seg_pct[seg]:5.2f}%)")
print()

print("Cross-tab: strategic segment × priority level")
print("(this should be PERFECTLY DIAGONAL within tier blocks)")
print(pd.crosstab(
    df_customers_agg['strategic_target_segment'],
    df_customers_agg['priority_level']
))
print()

Strategic target segment distribution:
  Protect High-Value Repeat Customers            11,972  ( 5.81%)
  Protect Highly Engaged Customers               12,880  ( 6.25%)
  Grow Affluent Active Customers                  6,118  ( 2.97%)
  Grow Active Family Shoppers                    30,628  (14.85%)
  Grow Active Value-Conscious Customers           2,073  ( 1.01%)
  Cross-Sell Broad Basket Explorers              11,236  ( 5.45%)
  Premium Higher Price Buyers                    16,422  ( 7.96%)
  Upsell Frequent Low-Price Buyers               13,167  ( 6.39%)
  Maintain Stable Mid-Value Customers            24,428  (11.85%)
  Reactivate Low-Engagement Families             54,315  (26.34%)
  Reactivate Low-Engagement Customers            22,970  (11.14%)

Cross-tab: strategic segment × priority level
(this should be PERFECTLY DIAGONAL within tier blocks)
priority_level                         Low Priority  Maintain  \
strategic_target_segment                                        
Pro

In [17]:
# ============================================================
# 7E — SAVE CHECKPOINT
# ============================================================

df_customers_agg.to_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'customers_agg_step7.pkl')
)

print(f"✅ NOTEBOOK 7 COMPLETE — checkpoint saved: customers_agg_step7.pkl")
print(f"   Shape: {df_customers_agg.shape}")
print(f"   Total columns: {df_customers_agg.shape[1]}")

✅ NOTEBOOK 7 COMPLETE — checkpoint saved: customers_agg_step7.pkl
   Shape: (206209, 27)
   Total columns: 27
